In [66]:
import pennylane as qml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [67]:
df = pd.read_csv('CC GENERAL.csv')

In [68]:
df

,CUST_ID,BALANCE,BALANCE_FREQUENCY,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,PURCHASES_FREQUENCY,ONEOFF_PURCHASES_FREQUENCY,PURCHASES_INSTALLMENTS_FREQUENCY,CASH_ADVANCE_FREQUENCY,CASH_ADVANCE_TRX,PURCHASES_TRX,CREDIT_LIMIT,PAYMENTS,MINIMUM_PAYMENTS,PRC_FULL_PAYMENT,TENURE
0,C10001,40.900749,0.818182,95.40,0.00,95.40,0.000000,0.166667,0.000000,0.083333,0.000000,0,2,1000.0,201.802084,139.509787,0.000000,12
1,C10002,3202.467416,0.909091,0.00,0.00,0.00,6442.945483,0.000000,0.000000,0.000000,0.250000,4,0,7000.0,4103.032597,1072.340217,0.222222,12
2,C10003,2495.148862,1.000000,773.17,773.17,0.00,0.000000,1.000000,1.000000,0.000000,0.000000,0,12,7500.0,622.066742,627.284787,0.000000,12
3,C10004,1666.670542,0.636364,1499.00,1499.00,0.00,205.788017,0.083333,0.083333,0.000000,0.083333,1,1,7500.0,0.000000,NaN,0.000000,12
4,C10005,817.714335,1.000000,16.00,16.00,0.00,0.000000,0.083333,0.083333,0.000000,0.000000,0,1,1200.0,678.334763,244.791237,0.000000,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8945,C19186,28.493517,1.000000,291.12,0.00,291.12,0.000000,1.000000,0.000000,0.833333,0.000000,0,6,1000.0,325.594462,48.886365,0.500000,6
8946,C19187,19.183215,1.000000,300.00,0.00,300.00,0.000000,1.000000,0.000000,0.833333,0.000000,0,6,1000.0,275.861322,NaN,0.000000,6
8947,C19188,23.398673,0.833333,144.40,0.00,144.40,0.000000,0.833333,0.000000,0.666667,0.000000,0,5,1000.0,81.270775,82.418369,0.250000,6
8948,C19189,13.457564,0.833333,0.00,0.00,0.00,36.558778,0.000000,0.000000,0.000000,0.166667,2,0,500.0,52.549959,55.755628,0.250000,6


In [69]:
x_data = df.to_numpy()
x_data = x_data[:,1:]
x_data = np.array(x_data,dtype=float)
x_data = x_data[:,np.std(x_data,axis=0)>0]

In [70]:
x_mean = np.sum(x_data,axis=0)/x_data.shape[0]
x_std = np.std(x_data,axis=0)
x_data = (x_data-x_mean)/x_std
print(x_data)

[[-0.73198937 -0.24943448 -0.42489974 ... -0.52897879 -0.52555097
   0.36067954]
 [ 0.78696085  0.13432467 -0.46955188 ...  0.81864213  0.2342269
   0.36067954]
 [ 0.44713513  0.51808382 -0.10766823 ... -0.38380474 -0.52555097
   0.36067954]
 ...
 [-0.7403981  -0.18547673 -0.40196519 ... -0.5706145   0.32919999
  -4.12276757]
 [-0.74517423 -0.18547673 -0.46955188 ... -0.58053567  0.32919999
  -4.12276757]
 [-0.57257511 -0.88903307  0.04214581 ... -0.57686873 -0.52555097
  -4.12276757]]


In [71]:
#I am dividing the points into four clusters
k=4
def quantum_computing(point,centroid):
    dev = qml.device('default.qubit',wires=2)
    distances = np.array([np.linalg.norm(point-centre) for centre in centroid])
    def quantum_circuit(point,distances):
        qml.Hadamard(wires=0)
        qml.Hadamard(wires=1)
        H0 = qml.Hamiltonian([-1,-1],[qml.PauliX(0),qml.PauliX(1)])
        [d0,d1,d2,d3] = distances
        c_I = (d0+d1+d2+d3)/4
        c_0 = (d0+d1-d2-d3)/4
        c_1 = (d0-d1+d2-d3)/4
        c_01 = (d0-d1-d2+d3)/4
        coeffs = [c_I,c_0,c_1,c_01]
        H1 = qml.Hamiltonian(coeffs,[qml.Identity(0),qml.PauliZ(0),qml.PauliZ(1),qml.PauliZ(0)@qml.PauliZ(1)])
        params = []
        dt = 1/20
        for i in range(20):
            params.append((i+1)*dt)
            params.append(1-i*dt)
        params = np.array(params)
        for i in range(20):
            qml.ApproxTimeEvolution(H1,params[2*i],3)
            qml.ApproxTimeEvolution(H0,params[2*i+1],3)
        return qml.probs(wires=[0,1])
    circuit = qml.QNode(quantum_circuit,dev)
    ans = np.argmax(circuit(point,distances))
    return ans

In [72]:
centroid_indices = np.random.choice(x_data.shape[0],size=4,replace=False)
centroid = x_data[centroid_indices]
cluster_label = np.ones(x_data.shape[0])*-1
for i in range(4):
    cluster_label[centroid_indices[i]]=i
centroid

array([[ 1.74349641e+00,  5.18083823e-01,  1.20832040e+00,
         9.71253950e-01,  1.07179659e+00, -4.66785554e-01,
         1.26984323e+00,  2.67345108e+00,  1.59919919e+00,
        -6.75348858e-01, -4.76069817e-01,  1.50023242e+00,
         4.16339471e-04, -5.25550971e-01,  3.60679544e-01],
       [-6.68412010e-01, -2.93575277e+00,  2.58666595e-01,
         5.80440670e-01, -4.54576230e-01, -4.66785554e-01,
        -1.01412545e+00, -3.99319268e-01, -9.16995191e-01,
        -6.75348858e-01, -4.76069817e-01, -5.11333250e-01,
         2.33563108e+00, -5.25550971e-01,  3.60679544e-01],
       [-4.37022361e-01,  5.18083823e-01, -1.74679214e-01,
         2.26308821e-02, -4.54576230e-01, -4.66785554e-01,
        -1.01412545e+00, -3.99319268e-01, -9.16995191e-01,
        -6.75348858e-01, -4.76069817e-01, -5.51564563e-01,
        -3.46743628e-01, -5.25550971e-01,  3.60679544e-01],
       [ 8.29597674e-03,  5.18083823e-01, -4.69551882e-01,
        -3.56934022e-01, -4.54576230e-01, -3.06871105

In [73]:
for i in range(x_data.shape[0]):
    cluster_label[i]=quantum_computing(x_data[i],centroid)
    new_centroid = []
    for j in range(4):
        cluster = x_data[np.where(cluster_label==j)[0]]
        new_point = np.mean(cluster,axis=0)
        new_centroid.append(new_point)
    centroid = np.array(new_centroid)

In [74]:
cluster_label

array([2., 3., 0., ..., 2., 3., 2.], shape=(8950,))

In [75]:
from sklearn.metrics import silhouette_score

In [76]:
score = silhouette_score(x_data, cluster_label)
print(f'score is {score}')

score is 0.17840692367267544
